In [1]:
import os
import json
import email
from email import policy
import glob
from pathlib import Path

DATASET_ROOT = Path(r"C:\Users\lafri\Downloads\Enron_Mail\maildir") # root folder
OUTPUT_FILE = "phase1_enron_clean.jsonl" # json output

# limit processing to business emails
# avoid junk like 'deleted_items'
RELEVANT_FOLDERS = ['inbox', 'sent', 'sent_items', 'notes_inbox', 'discussion_threads']

In [2]:
def parse_enron_email(file_path):
    """
    Parses a raw Enron email file using the Python email library.
    Returns: subject, body, sender, date
    """
    try:
        # Enron files are binary/latin-1, not perfect UTF-8
        with open(file_path, 'rb') as f:

            # Understands MIME, headers and handles multipart emails
            msg = email.message_from_binary_file(f, policy=policy.default)
            
        # Extract Text Body
        body = ""
        if msg.is_multipart():
            # Iterate parts to find text/plain
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        part_body = part.get_content()
                        if part_body:
                            body += part_body
                    except Exception:
                        continue
        else:
            # Single part message
            try:
                body = msg.get_content()
            except Exception:
                # for simple payload extraction
                body = msg.get_payload(decode=True).decode('utf-8', errors='ignore')

        # Removing forwarded headers
        # remove lines starting with ">" or "Original Message"        
        return {
            "subject": msg.get("Subject", ""),
            "from": msg.get("From", ""),
            "to": msg.get("To", ""),
            "date": msg.get("Date", ""),
            "body": body.strip()
        }
        
    except Exception as e:
        # print(f"Error parsing {file_path}: {e}")
        return None

In [3]:
print(f"Scanning {DATASET_ROOT} for Enron emails...")

if not os.path.exists(DATASET_ROOT):
    print(f"[!] Error: Path '{DATASET_ROOT}' not found.")
else:
    processed_count = 0
    skipped_count = 0
    
    # Open output file
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        
        # employee loop...........user folders (allen-p, arnold-j)
        for user_folder in os.listdir(DATASET_ROOT):
            user_path = os.path.join(DATASET_ROOT, user_folder)
            if not os.path.isdir(user_path): continue
            
            # mailbox loop.......subfolders (inbox, sent, etc.)
            for mail_folder in os.listdir(user_path):
                # Filter relevant folders only
                if mail_folder.lower() not in RELEVANT_FOLDERS:
                    continue
                
                folder_path = os.path.join(user_path, mail_folder)
                if not os.path.isdir(folder_path): continue
                
                # Process files in the folder
                files = os.listdir(folder_path)
                
                for filename in files:
                    file_path = os.path.join(folder_path, filename)
                    
                    data = parse_enron_email(file_path)
                    
                    if data and data['body']:
                        # Create a "Chat" style input for the LLM
                        # We combine Subject + Body as the "User Input"
                        full_text = f"Subject: {data['subject']}\n\n{data['body']}"
                        
                        entry = {
                            "source": "Enron",
                            "id": f"{user_folder}_{filename}",
                            "folder": mail_folder, # Hints at intent (Sent = Action, Inbox = Request)
                            "input": full_text,
                            "output": "TO_BE_GENERATED"
                        }
                        
                        f_out.write(json.dumps(entry) + "\n")
                        processed_count += 1
                        
                        if processed_count % 1000 == 0:
                            print(f"  Processed {processed_count} emails...")
                    else:
                        skipped_count += 1

    print("-" * 30)
    print(f"Done! Saved {processed_count} emails to {OUTPUT_FILE}")
    print(f"Skipped {skipped_count} empty/unreadable files.")

Scanning C:\Users\lafri\Downloads\Enron_Mail\maildir for Enron emails...
  Processed 1000 emails...
  Processed 2000 emails...
  Processed 3000 emails...
  Processed 4000 emails...
  Processed 5000 emails...
  Processed 6000 emails...
  Processed 7000 emails...
  Processed 8000 emails...
  Processed 9000 emails...
  Processed 10000 emails...
  Processed 11000 emails...
  Processed 12000 emails...
  Processed 13000 emails...
  Processed 14000 emails...
  Processed 15000 emails...
  Processed 16000 emails...
  Processed 17000 emails...
  Processed 18000 emails...
  Processed 19000 emails...
  Processed 20000 emails...
  Processed 21000 emails...
  Processed 22000 emails...
  Processed 23000 emails...
  Processed 24000 emails...
  Processed 25000 emails...
  Processed 26000 emails...
  Processed 27000 emails...
  Processed 28000 emails...
  Processed 29000 emails...
  Processed 30000 emails...
  Processed 31000 emails...
  Processed 32000 emails...
  Processed 33000 emails...
  Processed 